In [ ]:
from pathlib import Path
from io import StringIO

import pandas as pd

current_path = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (current_path, *current_path.parents)
        if (candidate / "data").is_dir() and (candidate / "src").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run this notebook from within the project directory.")
data_dir = project_root / "data"
input_csv_path = data_dir / "sample_cleaned.csv"

print("pandas_version:", pd.__version__)
print("project_root:", project_root)
print("data_dir_exists:", data_dir.exists())
print("input_csv_path:", input_csv_path)
print("input_csv_exists:", input_csv_path.exists())

In [ ]:
if input_csv_path.exists():
    previous_sample_df = pd.read_csv(input_csv_path)
    print("previous_sample_row_count:", len(previous_sample_df))
    print("previous_sample_column_count:", len(previous_sample_df.columns))
    display(previous_sample_df)
else:
    print("previous sample CSV is not available in this environment.")

In [ ]:
date_sample_csv_text = """company_name,ticker,sector,report_date,revenue_usd_m_text,operating_income_usd_m_text,net_income_usd_m_text,unit,frequency,source_note
Apple Inc.,AAPL,consumer technology,2024-12-31,"1,000",300,250,USD million,annual_sample,class_sample
Microsoft Corporation,MSFT,software and cloud,2024-12-31,900,320,260,USD million,annual_sample,class_sample
NVIDIA Corporation,NVDA,semiconductor,2024-12-31,,280,220,USD million,annual_sample,class_sample
Apple Inc.,AAPL,consumer technology,2023-12-31,950,290,230,USD million,annual_sample,class_sample
Microsoft Corporation,MSFT,software and cloud,2023-12-31,880,,240,USD million,annual_sample,class_sample
NVIDIA Corporation,NVDA,semiconductor,2023-12-31,650,220,not available,USD million,annual_sample,class_sample
"""

raw_date_df = pd.read_csv(StringIO(date_sample_csv_text))

print("raw_row_count:", len(raw_date_df))
print("raw_column_count:", len(raw_date_df.columns))
raw_date_df

In [ ]:
raw_date_df["report_date"] = pd.to_datetime(raw_date_df["report_date"])

print("date_dtype:", raw_date_df["report_date"].dtype)
print("min_report_date:", raw_date_df["report_date"].min().date())
print("max_report_date:", raw_date_df["report_date"].max().date())
raw_date_df[["ticker", "report_date"]]

In [ ]:
numeric_source_columns = [
    "revenue_usd_m_text",
    "operating_income_usd_m_text",
    "net_income_usd_m_text",
]

numeric_clean_columns = [
    "revenue_usd_m",
    "operating_income_usd_m",
    "net_income_usd_m",
]

for source_column, clean_column in zip(numeric_source_columns, numeric_clean_columns):
    raw_date_df[clean_column] = pd.to_numeric(
        raw_date_df[source_column].astype("string").str.replace(",", "", regex=False),
        errors="coerce",
    )

raw_date_df[
    [
        "ticker",
        "report_date",
        "revenue_usd_m_text",
        "revenue_usd_m",
        "operating_income_usd_m_text",
        "operating_income_usd_m",
        "net_income_usd_m_text",
        "net_income_usd_m",
    ]
]

In [ ]:
clean_columns = [
    "company_name",
    "ticker",
    "sector",
    "report_date",
    "revenue_usd_m",
    "operating_income_usd_m",
    "net_income_usd_m",
    "unit",
    "frequency",
    "source_note",
]

date_cleaned_df = raw_date_df[clean_columns].copy()

print("date_cleaned_row_count:", len(date_cleaned_df))
print("date_cleaned_column_count:", len(date_cleaned_df.columns))
date_cleaned_df

In [ ]:
date_sorted_df = date_cleaned_df.sort_values(["report_date", "ticker"]).copy()

print("first_report_date:", date_sorted_df["report_date"].iloc[0].date())
print("last_report_date:", date_sorted_df["report_date"].iloc[-1].date())
date_sorted_df[["report_date", "ticker", "revenue_usd_m", "net_income_usd_m"]]

In [ ]:
date_indexed_df = date_sorted_df.set_index("report_date")

print("index_name:", date_indexed_df.index.name)
print("index_dtype:", date_indexed_df.index.dtype)
print("date_index_is_monotonic:", date_indexed_df.index.is_monotonic_increasing)
date_indexed_df

In [ ]:
metric_columns = ["revenue_usd_m", "operating_income_usd_m", "net_income_usd_m"]

missing_summary = date_indexed_df.isna().sum()
metric_missing_summary = date_indexed_df[metric_columns].isna().sum()

print("missing_summary:")
print(missing_summary)
print("metric_missing_summary:")
print(metric_missing_summary)

In [ ]:
missing_row_mask = date_indexed_df[metric_columns].isna().any(axis=1)
missing_row_df = date_indexed_df[missing_row_mask]

print("missing_row_count:", len(missing_row_df))
missing_row_df[["company_name", "ticker", *metric_columns, "source_note"]]

In [ ]:
complete_case_df = date_indexed_df.dropna(subset=metric_columns)

print("complete_case_row_count:", len(complete_case_df))
complete_case_df[["company_name", "ticker", *metric_columns]]

In [ ]:
latest_date_df = date_indexed_df.loc["2024-12-31"]

print("latest_date_row_count:", len(latest_date_df))
latest_date_df[["company_name", "ticker", *metric_columns]]

In [ ]:
metric_missing_rate = date_indexed_df[metric_columns].isna().mean()

print("metric_missing_rate:")
print(metric_missing_rate)